# Import Libraries

In [ ]:
import os
import time
import argparse
from typing import Tuple
import torch
import torch.utils.data.dataloader
from torch.utils.data import DataLoader
from torchvision import transforms as T
from utils.set_seed import set_random_seed

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [ ]:
print(device)

In [ ]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [ ]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 8
learning_rate = 0.001
epochs = 10

set_random_seed(42) # seed for reproducibility

In [ ]:
from torchvision.transforms import AutoAugment, AutoAugmentPolicy

# my_augmentations = T.Compose([
#     T.RandomHorizontalFlip(p=0.5),
#     T.RandomVerticalFlip(p=0.5),
#     T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
#     T.RandomRotation(degrees=15),
#     T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
# ])

'''
# Might be wonky for some pipelines as AutoAugment performs some color transforms
Policy differs depending on the dataset used. The original authors of AutoAugment
provide the following policies:IMAGENET, CIFAR10, and SVHN. 

🔄 Geometric
- Rotate: small angle rotations (e.g. ±30°)
- ShearX / ShearY: slants the image along the x/y axis
- TranslateX / TranslateY: shifts the image horizontally or vertically
- Cutout: masks out a square region (like a blackout)
- Flip (not explicitly in every policy, but often used elsewhere)

🎨 Color/Intensity
- Brightness: changes image brightness
- Contrast: increases/decreases contrast
- Saturation: boosts or dulls the color intensity
- Sharpness: adjusts image sharpness
- Equalize: histogram equalization
- Posterize: reduces number of bits per channel (like retro pixelation)
- Solarize: inverts all pixel values above a certain threshold
- AutoContrast: maximizes contrast automatically
- Invert: inverts pixel colors
'''

# TODO: this must be updated once we include all 100K samples.
# Run testing.ipynb to recompute the mean and std values
bdd_mean = (0.2843, 0.3026, 0.2996)
bdd_std = (0.1918, 0.1945, 0.1989)

train_transforms = T.Compose([
    T.Resize((224, 224)),
    AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
    T.ToTensor(),
    T.Normalize(mean=bdd_mean, std=bdd_std),
])

# Load BDD100KPlus dataset
trainloader, testloader = get_dataloaders(
    dataset_name="Bdd100kPlus", 
    batch_size=batch_size, 
    train_transforms=train_transforms
)

In [ ]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in test set: {len(testloader)}")

# Initialize WeatherNet Model

In [ ]:
# wn_model: WeatherNet = WeatherNet()
wn_model: WeatherNet = WeatherNetPlusPlus()
# wn_model: WeatherNet = MtlWeatherNet()

# Print the model architecture
print(wn_model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(wn_model.parameters())

# Define loss function
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
saved_state = None # path to saved model state
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    wn_model.load_state_dict(torch.load(saved_state, weights_only=True))

# Model Training

In [ ]:
# Train the WeatherNet model and record losses
# Use the train function to train the wn_model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# epochs=1
wn_train_loss_log, wn_test_loss_log = trainer.train(wn_model, optimizer, criterion, trainloader, testloader, epochs, device)

# (TODO) Visualize Results